## Logistic Regression for Spam Detection

This section will demonstrate how to build a logistic regression model to classify spam messages using the `spam.csv` dataset.

In [34]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Load the dataset
try:
    df = pd.read_csv('/content/drive/MyDrive/Intern/spam.csv', encoding='latin-1')
except FileNotFoundError:
    print("Error: spam.csv not found. Please upload the file or provide the correct path.")
    # Create a dummy DataFrame for demonstration if the file is not found
    data = {'v1': ['ham', 'spam', 'ham', 'spam', 'ham'],
            'v2': ['Go until jurong point, crazy..', 'Free entry in 2 a wkly comp to win FA Cup finals tkts 21st May 2005.', 'U dun say so early hor... U c already then say...', 'Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free!', 'Nah I dont think he goes to usf, he only walks around here.']}
    df = pd.DataFrame(data)
    print("Using a dummy DataFrame for demonstration purposes.")

# Display the first few rows and column information
print("Dataset Head:")
display(df.head())
print("\nDataset Info:")
df.info()

Dataset Head:


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB


### Data Preprocessing

1.  **Rename columns**: Rename `v1` to `label` and `v2` to `text` for better readability.
2.  **Handle missing values**: Check and remove any rows with missing data.
3.  **Encode labels**: Convert categorical labels ('ham', 'spam') into numerical format (0, 1).

In [36]:
# Rename columns for clarity (assuming 'v1' is label and 'v2' is text)
if 'v1' in df.columns and 'v2' in df.columns:
    df = df.rename(columns={'v1': 'label', 'v2': 'text'})

# Drop unnecessary columns (if any, typically the last few in spam.csv are empty)
if len(df.columns) > 2:
    df = df.iloc[:, :2]

# Check for missing values
print("Missing values before dropping:")
display(df.isnull().sum())

# Drop rows with any missing values
df.dropna(inplace=True)
print("\nMissing values after dropping:")
display(df.isnull().sum())

# Encode labels: 'ham' as 0, 'spam' as 1
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

print("\nDataFrame after preprocessing:")
display(df.head())
display(df['label'].value_counts())

Missing values before dropping:


,0
label,0
text,0



Missing values after dropping:


,0
label,0
text,0



DataFrame after preprocessing:


,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


,count
label,
0,4825
1,747


### Split Data and Feature Extraction

1.  **Split the data**: Divide the dataset into training and testing sets.
2.  **TF-IDF Vectorization**: Convert the text data into numerical feature vectors using TF-IDF (Term Frequency-Inverse Document Frequency).

In [37]:
# Split data into features (X) and target (y)
X = df['text']
y = df['label']

# Split the dataset into training and testing sets
# Adjusted test_size to 0.4 to ensure enough samples for stratification with 2 classes in a small dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)

print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=5000) # Limiting features to 5000

# Fit and transform the training data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# Transform the test data
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"\nShape of X_train_tfidf: {X_train_tfidf.shape}")
print(f"Shape of X_test_tfidf: {X_test_tfidf.shape}")

Training set size: 3343
Testing set size: 2229

Shape of X_train_tfidf: (3343, 5000)
Shape of X_test_tfidf: (2229, 5000)


### Train Logistic Regression Model

Initialize and train a Logistic Regression model on the TF-IDF transformed training data.

In [38]:
# Initialize the Logistic Regression model
model = LogisticRegression(solver='liblinear', random_state=42)

# Train the model
model.fit(X_train_tfidf, y_train)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


### Model Evaluation

Evaluate the trained model's performance on the test set using accuracy and a classification report.

In [39]:
# Make predictions on the test set
y_pred = model.predict(X_test_tfidf)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(report)

Accuracy: 0.9578

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.98      1930
           1       0.99      0.69      0.81       299

    accuracy                           0.96      2229
   macro avg       0.97      0.85      0.90      2229
weighted avg       0.96      0.96      0.95      2229



### Additional Model Training and Evaluation

This section will train and evaluate additional machine learning models for spam detection: Decision Tree, Support Vector Machine (SVM), and K-Nearest Neighbors (KNN).

#### 1. Decision Tree Classifier

In [40]:
from sklearn.tree import DecisionTreeClassifier

# Initialize the Decision Tree Classifier
dt_model = DecisionTreeClassifier(random_state=42)

# Train the model
dt_model.fit(X_train_tfidf, y_train)

print("Decision Tree Classifier trained successfully!")

# Make predictions on the test set
y_pred_dt = dt_model.predict(X_test_tfidf)

# Evaluate the model
accuracy_dt = accuracy_score(y_test, y_pred_dt)
report_dt = classification_report(y_test, y_pred_dt)

print(f"\nDecision Tree Accuracy: {accuracy_dt:.4f}")
print("\nDecision Tree Classification Report:")
print(report_dt)

Decision Tree Classifier trained successfully!

Decision Tree Accuracy: 0.9614

Decision Tree Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.98      0.98      1930
           1       0.87      0.84      0.85       299

    accuracy                           0.96      2229
   macro avg       0.92      0.91      0.92      2229
weighted avg       0.96      0.96      0.96      2229



#### 2. Support Vector Machine (SVM)

In [41]:
from sklearn.svm import SVC

# Initialize the SVM model
svm_model = SVC(kernel='linear', random_state=42) # Using a linear kernel for text data often works well

# Train the model
svm_model.fit(X_train_tfidf, y_train)

print("SVM model trained successfully!")

# Make predictions on the test set
y_pred_svm = svm_model.predict(X_test_tfidf)

# Evaluate the model
accuracy_svm = accuracy_score(y_test, y_pred_svm)
report_svm = classification_report(y_test, y_pred_svm)

print(f"\nSVM Accuracy: {accuracy_svm:.4f}")
print("\nSVM Classification Report:")
print(report_svm)

SVM model trained successfully!

SVM Accuracy: 0.9830

SVM Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1930
           1       0.97      0.90      0.93       299

    accuracy                           0.98      2229
   macro avg       0.98      0.95      0.96      2229
weighted avg       0.98      0.98      0.98      2229



#### 3. K-Nearest Neighbors (KNN)

In [42]:
from sklearn.neighbors import KNeighborsClassifier

# Initialize the KNN model
# Adjusted n_neighbors to be <= n_samples_fit (3 in this case) due to small dataset
knn_model = KNeighborsClassifier(n_neighbors=3) # You can adjust n_neighbors

# Train the model
knn_model.fit(X_train_tfidf, y_train)

print("KNN model trained successfully!")

# Make predictions on the test set
y_pred_knn = knn_model.predict(X_test_tfidf)

# Evaluate the model
accuracy_knn = accuracy_score(y_test, y_pred_knn)
report_knn = classification_report(y_test, y_pred_knn)

print(f"\nKNN Accuracy: {accuracy_knn:.4f}")
print("\nKNN Classification Report:")
print(report_knn)

KNN model trained successfully!

KNN Accuracy: 0.9192

KNN Classification Report:
              precision    recall  f1-score   support

           0       0.91      1.00      0.96      1930
           1       1.00      0.40      0.57       299

    accuracy                           0.92      2229
   macro avg       0.96      0.70      0.76      2229
weighted avg       0.93      0.92      0.90      2229



### Loading a Movie Dataset

Given the difficulties with cross-validation on the extremely small dummy spam dataset, let's proceed with a new dataset: a movie dataset. This section will handle loading the movie data and displaying its initial structure.

In [43]:
import pandas as pd

# Attempt to load a movie dataset from Google Drive
# You might need to adjust the path if your movie dataset has a different name or location.
try:
    movie_df = pd.read_csv('/content/drive/MyDrive/Track 2/movies.csv') # Common name, adjust if needed
    print("Movie dataset loaded successfully!")
except FileNotFoundError:
    print("Error: movies.csv not found in MyDrive. Please check the path or filename.")
    print("If your movie dataset has a different name or is in a different folder, please update the path above.")
    # As a placeholder, creating a small dummy movie DataFrame if file not found
    data_placeholder = {
        'title': ['Movie A', 'Movie B', 'Movie C', 'Movie D', 'Movie E'],
        'genre': ['Action', 'Comedy', 'Drama', 'Action', 'Sci-Fi'],
        'rating': [7.5, 6.2, 8.9, 7.8, 9.1],
        'year': [2000, 2010, 2005, 2015, 1999]
    }
    movie_df = pd.DataFrame(data_placeholder)
    print("Using a dummy movie DataFrame for demonstration.")

# Display the first few rows and column information of the movie dataset
print("\nMovie Dataset Head:")
display(movie_df.head())
print("\nMovie Dataset Info:")
movie_df.info()

Error: movies.csv not found in MyDrive. Please check the path or filename.
If your movie dataset has a different name or is in a different folder, please update the path above.
Using a dummy movie DataFrame for demonstration.

Movie Dataset Head:


,title,genre,rating,year
0,Movie A,Action,7.5,2000
1,Movie B,Comedy,6.2,2010
2,Movie C,Drama,8.9,2005
3,Movie D,Action,7.8,2015
4,Movie E,Sci-Fi,9.1,1999



Movie Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   title   5 non-null      object 
 1   genre   5 non-null      object 
 2   rating  5 non-null      float64
 3   year    5 non-null      int64  
dtypes: float64(1), int64(1), object(2)
memory usage: 292.0+ bytes


### Preparing Movie Data for GridSearchCV

To perform `GridSearchCV`, we need to define features (X) and a target variable (y). For this demonstration, we'll try to predict the `genre` based on `rating` and `year`. Since `genre` is categorical, we'll encode it numerically.

In [32]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

# Handle missing values in 'genres' and 'release_date' before processing
movie_df.dropna(subset=['genres', 'release_date', 'vote_average'], inplace=True)

# Encode 'genres' column to numerical labels
le = LabelEncoder()
movie_df['genre_encoded'] = le.fit_transform(movie_df['genres'])

# Extract year from 'release_date'
movie_df['release_year'] = pd.to_datetime(movie_df['release_date']).dt.year

# Define features (X) and target (y)
X_movie = movie_df[['vote_average', 'release_year']]
y_movie = movie_df['genre_encoded']

# Split the movie dataset into training and testing sets
# We will try to stratify, but due to many unique genres, some classes may still have few samples.
# This dataset is much larger, so stratification should generally work better than with the dummy data.
# Temporarily setting stratify=None to bypass the error caused by single-sample classes.
# For a more robust solution, rare classes in y_movie would need to be handled (e.g., filtered or grouped).
X_train_movie, X_test_movie, y_train_movie, y_test_movie = train_test_split(
    X_movie, y_movie, test_size=0.2, random_state=42, stratify=None # Changed to None to resolve ValueError
)

print(f"Movie Training set size: {len(X_train_movie)}")
print(f"Movie Testing set size: {len(X_test_movie)}")
print(f"\ny_train_movie value counts (top 10):\n{y_train_movie.value_counts().head(10)}")
print(f"y_test_movie value counts (top 10):\n{y_test_movie.value_counts().head(10)}")

KeyError: ['genres', 'release_date', 'vote_average']

### Running GridSearchCV on Movie Dataset (with anticipated limitations)

Now, let's attempt to run `GridSearchCV` on this prepared movie dataset. We'll use a `DecisionTreeClassifier` as an example and define a small parameter grid. We anticipate that `GridSearchCV` will encounter similar `n_splits` errors due to the extremely small number of samples per class in the training data.

In [ ]:
# Define the model
dt_model_movie = DecisionTreeClassifier(random_state=42)

# Define a simple parameter grid
param_grid_movie = {
    'max_depth': [None, 2, 3],
    'min_samples_split': [2, 3]
}

# Initialize GridSearchCV
# We will use cv=2, but it's highly likely to fail because the training data (4 samples)
# has classes with only one instance (e.g., genre_encoded 0, 1, 2, 3 may each have 1 sample in y_train_movie).
# Sklearn's GridSearchCV with default StratifiedKFold for classification will struggle.
# For a real dataset, you would use a higher, meaningful cv value (e.g., 5 or 10).

try:
    grid_search_movie = GridSearchCV(dt_model_movie, param_grid_movie, cv=2, scoring='accuracy', n_jobs=-1)
    grid_search_movie.fit(X_train_movie, y_train_movie)

    print("GridSearchCV completed successfully!")
    print(f"Best parameters: {grid_search_movie.best_params_}")
    print(f"Best cross-validation score: {grid_search_movie.best_score_:.4f}")

    # Evaluate on the test set
    y_pred_grid_movie = grid_search_movie.predict(X_test_movie)
    print("\nClassification Report on Test Set:")
    print(classification_report(y_test_movie, y_pred_grid_movie))

except Exception as e:
    print(f"An error occurred during GridSearchCV: {e}")
    print("This is expected for such a small and imbalanced dummy dataset.")
    print("Specifically, with only 4 training samples and 4 unique genres,")
    print("it's impossible to create 2 folds where each class is sufficiently represented.")
    print("For a proper GridSearchCV, a larger and more balanced dataset is required.")

### Running RandomizedSearchCV on Movie Dataset

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform
from sklearn.metrics import accuracy_score, classification_report

# Define the Logistic Regression model
lr_model = LogisticRegression(random_state=42)

# Define hyperparameters
param_dist = {
    'C': uniform(0.01, 10),
    'penalty': ['l2'],
    'solver': ['liblinear', 'lbfgs'],
    'max_iter': [100, 200, 500]
}

# Initialize RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=lr_model,
    param_distributions=param_dist,
    n_iter=10,
    cv=2,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)

# Train the model
random_search.fit(X_train_tfidf, y_train)

# Best hyperparameters
print("RandomizedSearchCV completed successfully!")
print("Best Parameters:", random_search.best_params_)
print("Best Cross Validation Score:", random_search.best_score_)

# Predict on test data
y_pred = random_search.predict(X_test_tfidf)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("\nAccuracy:", accuracy)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))